# ЗАДАНИЕ НА "ХОРОШО""

# Парсинг с помощью BeautifulSoup

Beautiful Soup - это библиотека для Python, которая позволяет парсить (анализировать) HTML и XML документы. Она предоставляет удобный способ искать, навигировать, и модифицировать дерево DOM (Document Object Model), представляющее HTML/XML документ.

# Задание

Вам необходимо собрать датасет, спарсив данные из этого сайта:

https://books.toscrape.com/

Всего на сайте 1000 книг. То есть длина датасета должна равняться количеству книг.
 
Итоговая таблица должна содержать следующие столбцы:

| Название столбца | Описание | 
|--|--|
|id| Идентификатор книги |
|book_name| Название книги |
|price| Цена в £ |
|stock| Наличие книги. 1 или 0|
|url| Ссылка на книгу |

**Примечание по столбцам:**
- `id` - заполняется разработчиком датасета. Первая спарсенная книга имеет `id` = `0`.
- `url` - должна содержать полную ссылку. Не только конец ссылки, указанный на сайте. То есть по данному url можно перейти одним кликом.

## Импорт библиотек

In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

## Cоздание датасета и парсинг данных

In [2]:
page_tpl = 'https://books.toscrape.com/catalogue/page-{}.html'
rows = []

for page in range(1, 51):
    page_url = page_tpl.format(page)
    r = requests.get(page_url, timeout=10)
    soup = BeautifulSoup(r.content, 'html.parser')

    for book in soup.find_all('article', class_='product_pod'):
        a = book.find('h3').find('a')
        price_text = book.find('p', class_='price_color').text
        availability = book.find('p', class_='availability').text

        rows.append({
            'id': len(rows),
            'book_name': a['title'],
            'price': float(re.sub(r'[^\d.]', '', price_text)),
            'stock': int('In stock' in availability),
            'url': urljoin(page_url, a['href']),
        })

df = pd.DataFrame(rows)

## Итоговый датасет

In [3]:
df.shape

(1000, 5)

In [4]:
display(
    df.head(),
    df.tail()
)

,id,book_name,price,stock,url
0,0,A Light in the Attic,51.77,1,https://books.toscrape.com/catalogue/a-light-i...
1,1,Tipping the Velvet,53.74,1,https://books.toscrape.com/catalogue/tipping-t...
2,2,Soumission,50.10,1,https://books.toscrape.com/catalogue/soumissio...
3,3,Sharp Objects,47.82,1,https://books.toscrape.com/catalogue/sharp-obj...
4,4,Sapiens: A Brief History of Humankind,54.23,1,https://books.toscrape.com/catalogue/sapiens-a...


,id,book_name,price,stock,url
995,995,Alice in Wonderland (Alice's Adventures in Won...,55.53,1,https://books.toscrape.com/catalogue/alice-in-...
996,996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,1,https://books.toscrape.com/catalogue/ajin-demi...
997,997,A Spy's Devotion (The Regency Spies of London #1),16.97,1,https://books.toscrape.com/catalogue/a-spys-de...
998,998,1st to Die (Women's Murder Club #1),53.98,1,https://books.toscrape.com/catalogue/1st-to-di...
999,999,"1,000 Places to See Before You Die",26.08,1,https://books.toscrape.com/catalogue/1000-plac...


# ДОПОЛНИТЕЛЬНОЕ ЗАДАНИЕ НА "ОТЛИЧНО"

Так, мы спарсили данные о книгах. Но данные какие-то неполные. Часть названия стирается из-за отображения и нет ни полного названия книги, ни описания этой книги, ни жанра.

Вам необходимо дополнить датасет, спарсив дополнительные данные из того же сайта:

https://books.toscrape.com/
 
Итоговая таблица должна содержать следующие столбцы:

| Название столбца | Описание | 
|--|--|
|id| Идентификатор книги |
|book_name| Название книги - только полное название|
|genre| жанр книги |
|desc| описание |
|price| Цена в £ |
|stock| Наличие книги. 1 или 0|
|url| Ссылка на книгу |
| num_of_rev | количество отзывов|

## Парсинг данных и обогащение датасета

In [9]:
from concurrent.futures import ThreadPoolExecutor

session = requests.Session()


def parse_book_page(url):
    for _ in range(3):
        try:
            r = session.get(url, timeout=10)
            break
        except requests.RequestException:
            continue
    else:
        raise RuntimeError(f'Не удалось загрузить {url}')

    soup = BeautifulSoup(r.content, 'html.parser')

    name = soup.find('h1').text.strip()
    # хлебные крошки: Home > Books > Genre > Title
    genre = soup.find('ul', class_='breadcrumb').find_all('li')[2].text.strip()

    # у части книг описания нет
    desc_header = soup.find('div', id='product_description')
    desc = desc_header.find_next_sibling('p').text.strip() if desc_header else ''

    num_of_rev = 0
    for row in soup.find('table', class_='table-striped').find_all('tr'):
        if row.find('th').text == 'Number of reviews':
            num_of_rev = int(row.find('td').text)

    return name, genre, desc, num_of_rev


with ThreadPoolExecutor(max_workers=8) as pool:
    details = list(pool.map(parse_book_page, df['url']))

df['book_name'] = [d[0] for d in details]
df['genre'] = [d[1] for d in details]
df['desc'] = [d[2] for d in details]
df['num_of_rev'] = [d[3] for d in details]

df = df[['id', 'book_name', 'genre', 'desc', 'price', 'stock', 'url', 'num_of_rev']]

## Итоговый датасет

In [10]:
df.shape

(1000, 8)

In [11]:
display(
    df.head(),
    df.tail()
)

,id,book_name,genre,desc,price,stock,url,num_of_rev
0,0,A Light in the Attic,Poetry,It's hard to imagine a world without A Light i...,51.77,1,https://books.toscrape.com/catalogue/a-light-i...,0
1,1,Tipping the Velvet,Historical Fiction,"""Erotic and absorbing...Written with starling ...",53.74,1,https://books.toscrape.com/catalogue/tipping-t...,0
2,2,Soumission,Fiction,"Dans une France assez proche de la nôtre, un h...",50.10,1,https://books.toscrape.com/catalogue/soumissio...,0
3,3,Sharp Objects,Mystery,"WICKED above her hipbone, GIRL across her hear...",47.82,1,https://books.toscrape.com/catalogue/sharp-obj...,0
4,4,Sapiens: A Brief History of Humankind,History,From a renowned historian comes a groundbreaki...,54.23,1,https://books.toscrape.com/catalogue/sapiens-a...,0


,id,book_name,genre,desc,price,stock,url,num_of_rev
995,995,Alice in Wonderland (Alice's Adventures in Won...,Classics,,55.53,1,https://books.toscrape.com/catalogue/alice-in-...,0
996,996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Sequential Art,High school student Kei Nagai is struck dead i...,57.06,1,https://books.toscrape.com/catalogue/ajin-demi...,0
997,997,A Spy's Devotion (The Regency Spies of London #1),Historical Fiction,"In England’s Regency era, manners and elegance...",16.97,1,https://books.toscrape.com/catalogue/a-spys-de...,0
998,998,1st to Die (Women's Murder Club #1),Mystery,"James Patterson, bestselling author of the Ale...",53.98,1,https://books.toscrape.com/catalogue/1st-to-di...,0
999,999,"1,000 Places to See Before You Die",Travel,"Around the World, continent by continent, here...",26.08,1,https://books.toscrape.com/catalogue/1000-plac...,0
